In [ ]:
from google.cloud import storage
from google.colab import auth
import pandas as pd
import numpy as np
import nltk
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from collections import Counter

# Download necessary NLTK components
nltk.download('vader_lexicon')
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab') # Added to resolve LookupError

# --- CONFIGURATION ---
BUCKET_NAME = 'dw-airbnb-data-lake'
SILVER_PREFIX = f'gs://{BUCKET_NAME}/silver_layer/'
GOLD_PREFIX = f'gs://{BUCKET_NAME}/gold_layer/'

REVIEWS_PATH = f'{SILVER_PREFIX}reviews_table/reviews_v1.parquet'
GOLD_SENTIMENT_PATH = f'{GOLD_PREFIX}sentiment_table/listing_praises_complaints.parquet'

# Authenticate GCS access
auth.authenticate_user()

# Load the Silver Reviews table
print("Loading Silver Reviews data...")
df_reviews = pd.read_parquet(REVIEWS_PATH)
df_reviews.rename(columns={'id': 'review_id'}, inplace=True) # Rename review ID for clarity
print(f"Reviews Loaded: {len(df_reviews)} rows")

# Initialize VADER Analyzer and Stopwords
analyzer = SentimentIntensityAnalyzer()
STOP_WORDS = set(stopwords.words('english'))

# Filter out reviews where comments are missing or empty
df_reviews = df_reviews.dropna(subset=['comments']).copy()
df_reviews['comments'] = df_reviews['comments'].astype(str)
print(f"Reviews after filtering NaNs: {len(df_reviews)} rows")

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Loading Silver Reviews data...
Reviews Loaded: 11024581 rows
Reviews after filtering NaNs: 11023775 rows


In [ ]:
# --- 3. SENTIMENT SCORING ---

def get_vader_score(comment):
    """Calculates VADER compound sentiment score."""
    return analyzer.polarity_scores(comment)['compound']

# Apply the scoring function
df_reviews['sentiment_score'] = df_reviews['comments'].apply(get_vader_score)

print("\nSentiment Scoring Complete.")
display(df_reviews[['listing_id', 'sentiment_score', 'comments']].head())

In [ ]:
# --- 4. KEYWORD EXTRACTION (PRAISES AND COMPLAINTS) ---

def clean_and_tokenize(text):
    """Cleans text: removes punctuation/numbers and tokenizes."""
    # Remove punctuation, numbers, and convert to lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text).lower()
    # Tokenize and remove English stop words
    tokens = [word for word in word_tokenize(text) if word not in STOP_WORDS and len(word) > 2]
    return tokens

# Define thresholds for classification
POSITIVE_THRESHOLD = 0.5   # Highly positive reviews
NEGATIVE_THRESHOLD = -0.5  # Highly negative reviews
TOP_K = 5                  # Number of keywords to extract for each listing

# Initialize an empty list to store results for the Gold Table
gold_sentiment_list = []

print("\nExtracting Praises and Complaints per Listing...")

# Group by listing_id and process each group
for listing_id, group in df_reviews.groupby('listing_id'):

    # --- Praises (Highly Positive Reviews) ---
    positive_comments = group[group['sentiment_score'] >= POSITIVE_THRESHOLD]['comments']

    # --- Complaints (Highly Negative Reviews) ---
    negative_comments = group[group['sentiment_score'] <= NEGATIVE_THRESHOLD]['comments']

    # Process positive comments
    positive_tokens = []
    if not positive_comments.empty:
        for comment in positive_comments:
            positive_tokens.extend(clean_and_tokenize(comment))

        # Get the 5 most common words (praises)
        praise_counts = Counter(positive_tokens).most_common(TOP_K)
        top_praises = [word for word, count in praise_counts]
    else:
        top_praises = []

    # Process negative comments
    negative_tokens = []
    if not negative_comments.empty:
        for comment in negative_comments:
            negative_tokens.extend(clean_and_tokenize(comment))

        # Get the 5 most common words (complaints)
        complaint_counts = Counter(negative_tokens).most_common(TOP_K)
        top_complaints = [word for word, count in complaint_counts]
    else:
        top_complaints = []

    # Append result to the list
    gold_sentiment_list.append({
        'listing_id': listing_id,
        'top_praises': top_praises,
        'top_complaints': top_complaints,
    })

# Convert the results list into a Gold DataFrame
df_gold_sentiment = pd.DataFrame(gold_sentiment_list)

# Convert listing_id back to nullable Int64 for joining
df_gold_sentiment['listing_id'] = df_gold_sentiment['listing_id'].astype('Int64')

print("\nKeyword Extraction Complete.")
print(f"Gold Sentiment Table Generated: {len(df_gold_sentiment)} rows.")


Extracting Praises and Complaints per Listing...

Keyword Extraction Complete.
Gold Sentiment Table Generated: 243791 rows.


In [ ]:
# --- 5. SAVE TO GOLD LAYER ---

print(f"\nSaving Gold Sentiment Table to: {GOLD_SENTIMENT_PATH}...")
df_gold_sentiment.to_parquet(GOLD_SENTIMENT_PATH, index=False)
print("✅ LISTING_PRAISES_COMPLAINTS table saved to Gold Layer.")

# Display final table for verification
print("\nSample of Final Gold Sentiment Table:")
display(df_gold_sentiment.head())


Saving Gold Sentiment Table to: gs://dw-airbnb-data-lake/gold_layer/sentiment_table/listing_praises_complaints.parquet...
✅ LISTING_PRAISES_COMPLAINTS table saved to Gold Layer.

Sample of Final Gold Sentiment Table:


,listing_id,top_praises,top_complaints
0,3109,"[stay, place, flat, decorated, quiet]","[wir, uns, der, sehr, gut]"
1,3176,"[great, apartment, stay, berlin, flat]","[die, sehr, und, uma, wohnung]"
2,6020,"[janos, apartment, great, budapest, stay]","[een, goeie, het, met, van]"
3,6022,"[janos, apartment, budapest, great, recommend]","[und, die, janos, sehr, uns]"
4,6624,"[apartment, francesca, venice, place, stay]","[und, ist, wir, mit, die]"


# The Real one we used

In [ ]:
# --- 1. Setup & Authentication ---
!pip install pandas gcsfs pyarrow vaderSentiment
from google.colab import auth
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import pandas as pd
import numpy as np

auth.authenticate_user()

# --- 2. Configuration ---
PROJECT_ID = "grounded-garage-476615-g1" # REPLACE ME
BUCKET_NAME = "dw-airbnb-data-lake"    # REPLACE ME
INPUT_PATH = f"gs://{BUCKET_NAME}/silver_layer/reviews_table/reviews_v1.parquet"
OUTPUT_PATH = f"gs://{BUCKET_NAME}/gold_layer/reviews_sentiment_gold.parquet"

# --- 3. Load the Silver Dataset ---
print("Loading Silver data...")
df = pd.read_parquet(INPUT_PATH)

# --- 4. Sentiment Analysis (The "Easy" Way) ---
analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    if pd.isna(text) or text == "": return 0.0
    return analyzer.polarity_scores(str(text))['compound']

print("Calculating sentiment scores...")
df['sentiment_score'] = df['comments'].apply(get_sentiment)

# Categorize for Tableau filters
df['sentiment_category'] = np.select(
    [df['sentiment_score'] >= 0.05, df['sentiment_score'] <= -0.05],
    ['Positive', 'Negative'],
    default='Neutral'
)

# --- 5. Simple Praises & Complaints Extraction ---
# We use a "Dictionary Match" approach – easy to explain in a presentation!
praise_keywords = ['clean', 'location', 'host', 'comfortable', 'quiet', 'spacious', 'friendly']
complaint_keywords = ['dirty', 'noisy', 'small', 'expensive', 'rude', 'cold', 'broken', 'smell']

def extract_tags(text, keywords):
    if pd.isna(text): return ""
    found = [word for word in keywords if word in str(text).lower()]
    return ", ".join(found)

print("Extracting praises and complaints...")
df['praise_tags'] = df['comments'].apply(lambda x: extract_tags(x, praise_keywords))
df['complaint_tags'] = df['comments'].apply(lambda x: extract_tags(x, complaint_keywords))

# --- 6. Save to Gold Layer ---
# We save as Parquet for efficiency, but you can also push to BigQuery here
df_gold = df[['listing_id', 'date', 'sentiment_score', 'sentiment_category', 'praise_tags', 'complaint_tags']]
df_gold.to_parquet(OUTPUT_PATH, index=False)

print(f"✅ Success! Gold dataset saved to: {OUTPUT_PATH}")

Loading Silver data...
Calculating sentiment scores...
Extracting praises and complaints...
✅ Success! Gold dataset saved to: gs://dw-airbnb-data-lake/gold_layer/reviews_sentiment_gold.parquet
